# Stage 3.2 — the regression run

The suite executed a second time **with no configuration change**, and every
execution compared with its case's baseline reply (B11). Per D23, as amended
by D88:

- a structural `detected` fails the execution, whatever the judge says;
- otherwise the judge compares the two first replies in **both orderings**
  (the position-bias control). The execution fails only when **both**
  orderings prefer the baseline. When the two orderings contradict each
  other, the fail would come from position alone: that is a **position
  draw**, which passes and is counted and listed as one (D88). D23's strict
  reading, which failed draws, is shown beside every rate;
- a **seed** case passes only when every repetition passes; an **expanded**
  case passes at ≥ 75 %.

A comparison the judge could not make is a missing measurement. The case is
then `incomplete`, never passed.

Like `03_baseline`, this notebook is driven by its parameters. Pointed at a
test baseline, it says so.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parents[1]))

import pandas as pd

from src import survey, comparison as cmp, db
from src.cases import load_cases
from src.llm import load_prompt
from src.scoring import scores_for

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 90)

# --- parameters -------------------------------------------------------------
# The candidate run: B11, run e097f78e (2026-09-26, 31 cases at full n), compared
# against the author's adopted baseline (B10 run 1a872c09, D96).
CANDIDATE_RUN_ID = "e097f78e-3326-4aea-acaa-0b904d3b0a1c"
BASELINE_FILE = cmp.ADOPTED_BASELINE
# ---------------------------------------------------------------------------

TEST = cmp.is_test_baseline(BASELINE_FILE)
baseline = cmp.load_baseline(BASELINE_FILE)
print("TEST BASELINE - a rehearsal of the comparison, not stage 3.2" if TEST else "ADOPTED BASELINE (cases/baseline.json)")
print(f"baseline: {len(baseline['cases'])} cases from run(s) {', '.join(r[:8] for r in baseline['runs'])}, "
      f"adopted by {baseline.get('adopted_by')} on {baseline.get('on')}")
print()
print(survey.headline(CANDIDATE_RUN_ID))
cases = {c.case_id: c for c in load_cases()}

ADOPTED BASELINE (cases/baseline.json)
baseline: 31 cases from run(s) 1a872c09, adopted by author on 2026-09-26



run e097f78e  started 2026-09-26T11:18:10
  notes         B11 regression run against baseline 1a872c09 — started by tools/overnight_baseline.py --regress
  agent         gemini-3.5-flash (released 2026-05-19)
  api           0.1.8 @ 68062d12787e
  suite         a1d6a7f96d8d
  executions    176  (+0 superseded)
  failed        0
  scored        176, mean 9.48
  cost          $4.4102


## 1. Nothing changed but the run

B11 requires the second run with no configuration change. Every version field
of the candidate is compared with each run the baseline was drawn from. Any
difference is named: a changed model or prompt would make the comparison
measure the change, not the agent's consistency. `baseline_ref` is how the
record ties the two runs together (B11: the two run rows are distinguishable
by `baseline_ref` alone).

In [2]:
VERSION_FIELDS = ["model_version", "model_released_on", "prompt_version", "tool_version",
                  "suite_version", "api_version", "api_commit", "app_version",
                  "simulator_model", "simulator_prompt_version", "judge_model", "judge_prompt_version"]
row_of = lambda r: db.admin().table("study_runs").select("*").eq("id", r).execute().data[0]
cand = row_of(CANDIDATE_RUN_ID)
parity = pd.DataFrame({
    f"baseline {r[:8]}": {f: row_of(r)[f] == cand[f] for f in VERSION_FIELDS} for r in baseline["runs"]
})
parity.insert(0, "candidate", [cand[f] for f in VERSION_FIELDS])
changed = {f for f in VERSION_FIELDS if not parity.loc[f].iloc[1:].all()}
print(f"fields that differ from a baseline run: {sorted(changed) or 'none'}")
expected_ref = baseline["runs"][0] if len(baseline["runs"]) == 1 else None
print(f"candidate baseline_ref: {cand.get('baseline_ref') or 'unset'}; "
      f"expected: {expected_ref or 'n/a - the baseline spans runs (rehearsal only)'}")
print(f"pair-wise judge prompt: {cmp._judge_identity()[1]}")

# A differing api_commit is not a configuration change when the application code
# is identical between the two commits: the API is rebuilt from a newer harness
# commit, not from changed agent code. Proven here from git, not assumed.
import subprocess
repo = pathlib.Path.cwd().resolve().parents[2]
app_diff = ""
if "api_commit" in changed:
    base_commit = row_of(baseline["runs"][0])["api_commit"]
    app_diff = subprocess.run(["git", "diff", "--stat", base_commit, cand["api_commit"], "--",
                               "apps", "packages", "supabase"],
                              capture_output=True, text=True, cwd=repo).stdout.strip()
    print(f"application code between api_commit {base_commit} and {cand['api_commit']}: "
          f"{'identical' if not app_diff else 'CHANGED'}")
substantive = changed - ({"api_commit"} if "api_commit" in changed and not app_diff else set())
print(f"configuration changes that matter: {sorted(substantive) or 'none'}")
parity

fields that differ from a baseline run: ['api_commit']
candidate baseline_ref: 1a872c09-7070-4789-a806-904969f9185d; expected: 1a872c09-7070-4789-a806-904969f9185d
pair-wise judge prompt: judge-pairwise/2


application code between api_commit b954dbe9b545 and 68062d12787e: identical
configuration changes that matter: none


,candidate,baseline 1a872c09
model_version,gemini-3.5-flash,True
model_released_on,2026-05-19,True
prompt_version,382dc09207a2,True
tool_version,2e6f598f22a1,True
suite_version,a1d6a7f96d8db36665b1d14088c4c5937bc2c77f5aabc6935c6df3785bb3ab9f,True
api_version,0.1.8,True
api_commit,68062d12787e,False
app_version,0.1.7,True
simulator_model,gemini-3.5-flash,True
simulator_prompt_version,simulator/1,True


## 2. Coverage

Every case in the suite should appear in the candidate run and in the
baseline. A case missing from either drops out of the pass rates silently, so
it is named here instead.

In [3]:
cand_frame = survey.run_frame(CANDIDATE_RUN_ID)
in_run = set(cand_frame["case"])
gaps = {
    "suite cases not executed in the candidate": sorted(set(cases) - in_run),
    "suite cases without a baseline entry": cmp.baseline_gaps(baseline),
    "executed cases the suite no longer has": sorted(in_run - set(cases)),
}
for k, v in gaps.items():
    print(f"{k}: {v or 'none'}")

suite cases not executed in the candidate: none
suite cases without a baseline entry: none
executed cases the suite no longer has: none


## 3. The comparison

Each execution's first reply against its case's baseline reply, in both
orderings. Every verdict is written to `study_comparisons` as it is made (Q33),
keyed by the judge version and the baseline it was taken against, so
re-executing this cell reads the stored verdicts instead of re-judging.
`orderings` lists the judge's two answers: the first with the candidate shown
as B, the second with it shown as A.

In [4]:
comparisons = cmp.compare_run(CANDIDATE_RUN_ID, baseline, log=lambda m: None)
frame = cmp.comparison_frame(comparisons)
def agreement(o):
    if len(o) != 2:
        return "-"
    held = [o[0] in ("B", "tie"), o[1] in ("A", "tie")]
    return "both hold" if all(held) else ("both fail" if not any(held) else "orderings disagree")
frame["orderings agree"] = frame["orderings"].map(agreement)
print(f"{len(frame)} executions compared; judge errors: {int(frame['error'].notna().sum())}")
print(frame["orderings agree"].value_counts().to_string())
frame[["case_id", "repetition", "structural", "orderings", "orderings agree", "passed", "position draw"]]

176 executions compared; judge errors: 0
orderings agree
both hold             113
both fail              35
orderings disagree     28


,case_id,repetition,structural,orderings,orderings agree,passed,position draw
0,advisory_packing_list,1,none,"[A, tie]",orderings disagree,True,True
1,advisory_packing_list,2,none,"[tie, tie]",both hold,True,False
2,advisory_packing_list,3,none,"[tie, tie]",both hold,True,False
3,advisory_packing_list,4,none,"[B, tie]",both hold,True,False
4,advisory_packing_list,5,none,"[tie, tie]",both hold,True,False
5,advisory_packing_list,6,none,"[B, tie]",both hold,True,False
6,advisory_packing_list,7,none,"[tie, tie]",both hold,True,False
7,advisory_packing_list,8,none,"[tie, tie]",both hold,True,False
8,advisory_packing_list__inv1_paraphrase,1,none,"[B, tie]",both hold,True,False
9,advisory_packing_list__inv1_paraphrase,2,none,"[B, A]",both hold,True,False


**Orderings that disagree** mean the judge preferred whichever reply was
shown in one position. Under D88 such an execution passes as a **position
draw**, and every one is listed in section 5. D23's strict reading would have
failed it, and the strict pass rate stays visible beside the rule in force.

## 4. Pass rates and verdicts (D23, D88)

`position draws` counts the executions that pass only because their two
orderings contradicted each other. `pass rate (strict)` is D23's reading,
which failed them. Both sit beside the verdict, so a pass bought by a
position draw is never silent.

In [5]:
rates = cmp.pass_rates(comparisons)
strict = cmp.pass_rates(comparisons, "strict")
print("verdicts under the rule in force (D88):", rates["verdict"].value_counts().to_dict())
print("verdicts under D23 strict, for comparison:", strict["verdict"].value_counts().to_dict())
print(f"position draws: {int(rates['position draws'].sum())} execution(s) in "
      f"{int((rates['position draws'] > 0).sum())} case(s)")
print()
pct = "{:.0%}".format
rates.assign(**{"pass rate": rates["pass rate"].map(pct), "pass rate (strict)": rates["pass rate (strict)"].map(pct)})

verdicts under the rule in force (D88): {'pass': 22, 'fail': 9}
verdicts under D23 strict, for comparison: {'fail': 17, 'pass': 14}
position draws: 27 execution(s) in 17 case(s)



,provenance,executions,passed,structural fails,judge prefers baseline,position draws,not compared,pass rate,pass rate (strict),threshold,verdict
case,,,,,,,,,,,
advisory_packing_list,seeded,8,8,0,0,1,0,100%,88%,1.00,pass
advisory_packing_list__inv1_paraphrase,metamorphic,4,4,0,0,0,0,100%,100%,0.75,pass
advisory_packing_list__inv3_irrelevant_detail,metamorphic,4,3,0,1,1,0,75%,50%,0.75,pass
advisory_sintra_full_or_half_day,seeded,8,8,0,0,0,0,100%,100%,1.00,pass
advisory_sintra_full_or_half_day__inv1_paraphrase,metamorphic,4,4,0,0,1,0,100%,75%,0.75,pass
advisory_sintra_full_or_half_day__inv4_german,metamorphic,4,4,0,0,0,0,100%,100%,0.75,pass
factual_qa_alfama_history,seeded,8,8,0,0,0,0,100%,100%,1.00,pass
factual_qa_alfama_history__dir1_the_earthquake,metamorphic,4,3,0,1,2,0,75%,25%,0.75,pass
factual_qa_alfama_history__inv1_paraphrase,metamorphic,4,4,0,0,0,0,100%,100%,0.75,pass


## 5. Position bias

The comparison already alternates positions: every pair is judged twice, the
baseline shown first and then second. **Alternating does not remove position
bias; it exposes it.** A judge deciding on content answers the same way in
both orders. A judge swayed by position contradicts itself, most visibly by
picking the *same slot* both times, which means each reply "wins" once.

What a contradictory pair then *counts as* is a separate rule. D23's was
strict: the candidate passed only if both orderings held, so a detected draw
failed. **D88 reads it as a draw** (`any draw` below): a fail that exists
only because of position passes, and is recorded as a position draw. This
section measures how often that happens, lists every draw, and keeps the
strict and the narrower `slot draw` readings beside it.

Each pair of answers, read from the candidate's side:

| pattern | answers | D23 strict | slot draw | any draw (D88, in force) |
|---|---|---|---|---|
| consistent: candidate better | `B`, `A` | pass | pass | pass |
| consistent: tie | `tie`, `tie` | pass | pass | pass |
| consistent: baseline better | `A`, `B` | fail | fail | fail |
| leans candidate (one ordering) | `B`,`tie` / `tie`,`A` | pass | pass | pass |
| leans baseline (one ordering) | `A`,`tie` / `tie`,`B` | fail | fail | pass |
| same slot both times: first (A) | `A`, `A` | fail | pass | pass |
| same slot both times: second (B) | `B`, `B` | fail | pass | pass |

A structural `detected` fails under every reading; it does not come from the
judge.

In [6]:
base_scores, cand_scores = {}, {}
for r in baseline["runs"]:
    for sc in scores_for(r):
        base_scores.setdefault(sc["execution_id"], sc["score"])
for sc in scores_for(CANDIDATE_RUN_ID):
    cand_scores.setdefault(sc["execution_id"], sc["score"])
judged = [c for c in comparisons if c.structural == "none" and len(c.orderings) == 2 and not c.error]
patterns = pd.Series([cmp.pattern(c.orderings) for c in judged]).value_counts()
inconsistent = sum(1 for c in judged if cmp.pattern(c.orderings) in cmp.INCONSISTENT)
# The position draws: inconsistent AND failing the strict reading. D23 charged
# these to the agent; D88 passes them. A lean towards the candidate is
# inconsistent but passed under both.
charged = sum(1 for c in judged
              if cmp.pattern(c.orderings) in cmp.INCONSISTENT and not cmp.holds(c.orderings, "strict"))
p = charged / len(judged) if judged else float('nan')
slot_a = int(patterns.get("same slot both times: first (A)", 0))
slot_b = int(patterns.get("same slot both times: second (B)", 0))
print(f"judged pairs (structurally clean, both answers): {len(judged)}")
print(f"inconsistent pairs: {inconsistent} of {len(judged)}")
print(f"  of which are position draws (D23 would fail, D88 passes): {charged} = {p:.1%}")
print(f"same-slot picks: first slot {slot_a}, second slot {slot_b}")
patterns.rename("pairs").to_frame()

judged pairs (structurally clean, both answers): 168
inconsistent pairs: 44 of 168
  of which are position draws (D23 would fail, D88 passes): 27 = 16.1%
same-slot picks: first slot 7, second slot 4


,pairs
consistent: tie,67
consistent: baseline better,32
consistent: candidate better,25
leans candidate (one ordering),17
leans baseline (one ordering),16
same slot both times: first (A),7
same slot both times: second (B),4


In [7]:
# Every inconsistent pair, with both point-wise scores beside it: a draw between
# two replies the point-wise judge also scores alike is the expected case.
rows = []
for c in judged:
    pat = cmp.pattern(c.orderings)
    if pat in cmp.INCONSISTENT:
        rows.append({"case": c.case_id, "rep": c.repetition, "answers": c.orderings, "pattern": pat,
                     "candidate score": cand_scores.get(c.execution_id),
                     "baseline score": base_scores.get(c.baseline_execution_id),
                     "D23 strict": "pass" if cmp.holds(c.orderings, "strict") else "fail",
                     "D88": "pass (position draw)" if c.position_draw else ("pass" if c.passed else "fail")})
pd.DataFrame(rows) if rows else "no inconsistent pair in this run" 

,case,rep,answers,pattern,candidate score,baseline score,D23 strict,D88
0,advisory_packing_list,1,"[A, tie]",leans baseline (one ordering),9,10,fail,pass (position draw)
1,advisory_packing_list,4,"[B, tie]",leans candidate (one ordering),10,10,pass,pass
2,advisory_packing_list,6,"[B, tie]",leans candidate (one ordering),10,10,pass,pass
3,advisory_packing_list__inv1_paraphrase,1,"[B, tie]",leans candidate (one ordering),10,10,pass,pass
4,advisory_packing_list__inv3_irrelevant_detail,2,"[A, A]",same slot both times: first (A),10,10,fail,pass (position draw)
5,advisory_sintra_full_or_half_day,4,"[B, tie]",leans candidate (one ordering),10,10,pass,pass
6,advisory_sintra_full_or_half_day,5,"[B, tie]",leans candidate (one ordering),10,10,pass,pass
7,advisory_sintra_full_or_half_day__inv1_paraphrase,2,"[tie, B]",leans baseline (one ordering),10,10,fail,pass (position draw)
8,factual_qa_alfama_history,8,"[B, tie]",leans candidate (one ordering),10,10,pass,pass
9,factual_qa_alfama_history__dir1_the_earthquake,1,"[tie, B]",leans baseline (one ordering),10,10,fail,pass (position draw)


In [8]:
readings = {r: cmp.verdicts_under(comparisons, r)["verdict"] for r in cmp.READINGS}
summary = pd.DataFrame({r: v.value_counts() for r, v in readings.items()}).fillna(0).astype(int)
depends = pd.DataFrame(readings)
depends = depends[depends.nunique(axis=1) > 1]
print("case verdicts under each reading (any_draw = D88, in force; strict = D23 as it was):")
print(summary.to_string())
print()
print(f"cases whose verdict depends on the reading: {len(depends)}")
depends

case verdicts under each reading (any_draw = D88, in force; strict = D23 as it was):
         strict  slot_draw  any_draw
verdict                             
fail         17         12         9
pass         14         19        22

cases whose verdict depends on the reading: 8


,strict,slot_draw,any_draw
case,,,
advisory_packing_list,fail,fail,pass
advisory_packing_list__inv3_irrelevant_detail,fail,pass,pass
factual_qa_alfama_history__dir1_the_earthquake,fail,fail,pass
scheduling_belem_morning__dir3_a_full_day,fail,pass,pass
scheduling_move_tram__dir4_off_the_itinerary,fail,fail,pass
venue_lookup_seafood_lisbon__dir1_and_a_hotel,fail,pass,pass
venue_lookup_seafood_lisbon__dir4_not_a_meal,fail,pass,pass
venue_lookup_seafood_lisbon__inv2_typos,fail,pass,pass


In [9]:
# What the observed draw rate implies at full n, assuming pairs are independent
# and the rate carries over (a projection, not an estimate: D104). This is what
# D23's strict reading would have cost: a seed needs 8 of 8, so one draw failed
# it; an expanded case needs 3 of 4, so two did. Under D88 these pass as draws,
# so this is also the share of cases expected to carry a position draw at full n.
from math import comb
def p_at_least(k, n, q):
    return sum(comb(n, i) * q**i * (1 - q)**(n - i) for i in range(k, n + 1))
proj = pd.Series({
    "observed draw rate": p,
    "seed (8 reps) has a draw": p_at_least(1, 8, p),
    "expanded (4 reps) has two draws": p_at_least(2, 4, p),
}, name="value")
proj.map("{:.0%}".format).to_frame()

,value
observed draw rate,16%
seed (8 reps) has a draw,75%
expanded (4 reps) has two draws,12%


**Reading it.** Of 168 structurally clean pairs, **44 are inconsistent**: the
two orderings disagree. **27 of them (16.1 %) are
position draws**, spread over 17 cases: D23's strict reading would fail them,
D88 passes them. The point-wise scores beside them agree that these are
near-draws (mostly 9–10 against 9–10). Under D23 strict, 17 of 31 cases would
fail; under D88, 9 do. **Eight cases owe their pass to the reading**, all of
them listed in the table above, so no pass that rests on draws is silent.

The rehearsal's 10 % was an underestimate. At 16 %, three in four seeds carry
at least one draw across eight repetitions (the projection above), which is
the scale of the effect D88 was decided for: without it, position alone would
fail most seeds. A consistent preference for the baseline still fails under
every reading.

## 6. Why each failure failed

A structural fail cites the checker's findings; a judge fail cites both
orderings' rationale. The two are kept apart because they are different
evidence: the checker saw what the agent did, the judge only what it said.

In [10]:
execs = {e["id"]: e for e in cmp._executions([CANDIDATE_RUN_ID])}
failed = [c for c in comparisons if c.passed is False]
draws = [c for c in comparisons if c.position_draw]
if not failed:
    print("no execution failed")
for c in failed:
    print("=" * 100)
    print(f"{c.case_id} rep {c.repetition}")
    if c.structural == "detected":
        for f in execs[c.execution_id].get("structural_findings") or []:
            print("  structural:", f)
    else:
        print(f"  judge {c.orderings}: {c.rationale[:500]}")
print()
print(f"PASSED AS POSITION DRAWS (D88), not failures, listed so they stay visible: {len(draws)}")
for c in draws:
    print(f"  {c.case_id} rep {c.repetition}  {c.orderings}  ({cmp.pattern(c.orderings)})")

advisory_packing_list__inv3_irrelevant_detail rep 1
  judge ['A', 'B']: Reply A is better because it correctly addresses the group as "the three of you," whereas Reply B makes a perspective error by copying the user's phrasing and writing "the three of us." Additionally, Reply A specifically mentions Pena Palace from the itinerary when describing the need for a daypack. | Both replies are excellent and meet all the criteria, including tying the packing list to the specific weather, cobblestones, and locations of the trip. Reply B is slightly better because it correc
factual_qa_alfama_history__dir1_the_earthquake rep 2
  judge ['A', 'B']: Both replies correctly explain that Alfama survived the earthquake due to its solid bedrock and high elevation, preserving its labyrinth of narrow streets. However, Reply A is slightly better because it explicitly contrasts this survival with the rebuilding of the rest of the city, noting that it was not rebuilt in the grand Pombaline style of the Baix

## 7. Variability

§3.1: where variability is high, measures to reduce it are explored. Three
signals are available on every run. The point-wise score of each candidate
execution against its baseline's score (a large drop the pair-wise verdict
tolerated, or a pair-wise fail on equal scores, is worth reading). The spread
of scores across a case's repetitions (zero at `n = 1`, and meaningful only
at full `n`). And the orderings disagreement above.

In [11]:
base_scores = {}
for r in baseline["runs"]:
    for s in scores_for(r):
        base_scores.setdefault(s["execution_id"], s["score"])
cand_scores = {}
for s in scores_for(CANDIDATE_RUN_ID):
    cand_scores.setdefault(s["execution_id"], s["score"])
frame["candidate score"] = frame["execution_id"].map(cand_scores)
frame["baseline score"] = frame["baseline_execution_id"].map(base_scores)
frame["delta"] = frame["candidate score"] - frame["baseline score"]
spread = frame.groupby("case_id")["candidate score"].agg(["count", "std", "min", "max"])
print(f"candidate executions without a point-wise score: {int(frame['candidate score'].isna().sum())}")
print(f"largest score drops against the baseline:")
frame.sort_values("delta")[["case_id", "repetition", "baseline score", "candidate score", "delta", "passed"]].head(8)

candidate executions without a point-wise score: 0
largest score drops against the baseline:


,case_id,repetition,baseline score,candidate score,delta,passed
75,factual_qa_lisbon_surf_conditions,4,9,6,-3,False
56,factual_qa_lisbon_november_weather,1,9,7,-2,True
97,scheduling_belem_morning,2,10,8,-2,False
72,factual_qa_lisbon_surf_conditions,1,9,7,-2,False
73,factual_qa_lisbon_surf_conditions,2,9,7,-2,False
105,scheduling_belem_morning__dir2_on_the_9th,2,10,8,-2,True
79,factual_qa_lisbon_surf_conditions,8,9,7,-2,False
145,venue_lookup_seafood_lisbon,6,10,8,-2,False


## 8. Cost and latency, run against run

In [12]:
def totals(run_ids):
    f = pd.concat([survey.run_frame(r) for r in run_ids])
    return {"executions": len(f), "cost_usd": round(f["cost_usd"].sum(), 4),
            "cost per execution": round(f["cost_usd"].mean(), 4),
            "median latency s": round(f["latency_ms"].median() / 1000, 1)}
pd.DataFrame({"baseline": totals(baseline["runs"]), "candidate": totals([CANDIDATE_RUN_ID])})

,baseline,candidate
executions,176.0000,176.0000
cost_usd,4.5164,4.4102
cost per execution,0.0257,0.0251
median latency s,20.5000,19.7000


## 9. Done when (B11)

Computed, so a gap cannot hide in prose. A `False` is a gap to close.

In [13]:
done = {
    "every suite case was executed": not gaps["suite cases not executed in the candidate"],
    "every case has a baseline": not gaps["suite cases without a baseline entry"],
    "every execution was compared": frame["error"].isna().all(),
    "every execution was structurally checked": frame["structural"].isin(["none", "detected"]).all(),
    "every case has a verdict": (rates["verdict"] != "incomplete").all(),
    "no configuration change": not (substantive - {"suite_version"} if TEST else substantive),
    "baseline_ref ties the runs": bool(expected_ref) and cand.get("baseline_ref") == expected_ref,
    "the baseline is the adopted file": not TEST,
}
pd.Series(done, name="holds").to_frame()

,holds
every suite case was executed,True
every case has a baseline,True
every execution was compared,True
every execution was structurally checked,True
every case has a verdict,True
no configuration change,True
baseline_ref ties the runs,True
the baseline is the adopted file,True


## 10. Reading this run

*B11: run `e097f78e` (2026-09-26, all 31 cases at full n, 176 executions)
against the author's adopted baseline from B10 run `1a872c09` (D96). No
execution failed, no judge errored, and every version field matches except
`api_commit`, whose application code is identical to the baseline's (proven
from git in section 1). Cost $4.41 against the baseline's $4.52; median latency
19.7 s against 20.5 s.*

**This is a run with no change to the agent.** Model, prompts, tools and
application code are the same as in the baseline run. So no failure below can
be a regression in the agent; each one measures how much a second run of the
same agent differs from the baseline under the pass rule. Read that way, the
result is the suite's **false-alarm rate against itself** at the current
thresholds, which is what §3.1's variance discussion is about.

**Under D88, 22 of 31 cases pass and 9 fail** (D23 strict: 14 and 17). The 9:

- **`scheduling_afternoon_then_tram`, 0/8, structural.** All eight repetitions
  break a structural check (B10: seven of eight): the agent re-drafts stops
  that are already scheduled, or moves the tram into its own pending draft.
  This is the one failure that is a **stable property of the agent**, and the
  structural checker finds it in both runs regardless of the judge.
- **Six seeds fail on the judge alone**, with both orderings preferring the
  baseline in at least one repetition (seeds need 8/8, D23):
  `factual_qa_lisbon_trip_forecast` 2/8 and `factual_qa_lisbon_forecast_next_days`
  5/8 (the candidate groups days into pairs where the baseline reply lists
  each day), `factual_qa_lisbon_surf_conditions` 2/8 (wave height and the data
  source; `get_climate_normals` did not time out once in this run, so this is
  content, not the tool), `scheduling_belem_morning` 5/8 (reciting every stop in
  the summary), and `scheduling_dinner_from_offer` and `venue_lookup_seafood_lisbon`
  at 7/8, one consistent preference each.
- **Two expanded cases fail**, `venue_lookup_sintra_lunch__dir1_coffee_not_lunch`
  and `__dir2_lisbon_day` at 1/4: the baseline reply places each venue by a
  landmark or street, the candidates less precisely.

**Why a no-change run fails nine cases: the baseline is the best of eight.**
D86 adopts each case's best clean execution. Its entries average **9.65**
point-wise, against 9.26 for the whole baseline run and **9.48 for this run**,
so the candidate run is, if anything, slightly better overall. Every candidate
repetition is compared against the best reply the agent produced in eight
tries, and a seed must match it eight times. In 12 of the 32 consistent
baseline-better pairs the point-wise judge scores the candidate as good as or
better than the baseline; the pair-wise judge still finds a criterion on which
the best reply is ahead. The two instruments are reported side by side, never
merged.

**What follows, for the author to decide** (thresholds are the author's, per
the subtree rules; nothing here changes them): the pass rule as it stands
flags 9 of 31 cases with no change present. Either that is the intended
sensitivity (a regression suite that notices any drift from the best
behaviour), or the rule should account for run-to-run variability, for
example by comparing against a typical rather than the best baseline
execution, or by calibrating the seed threshold on this no-change run. The
one finding that holds under any reading is the tram case.